# Chapter 1: Tiny VLA

Build a complete VLA pipeline from scratch on MiniPushT.

**Runtime:** CPU or T4 GPU (~2-3 min)

In [ ]:
# Install dependencies
!pip install torch numpy gymnasium matplotlib -q

In [ ]:
# Clone repo (if running on Colab)
import os
if not os.path.exists('vla-from-scratch'):
    !git clone https://github.com/FanFeast/vla-from-scratch
os.chdir('vla-from-scratch/chapters/01_tiny_vla')

In [ ]:
from mini_pusht import MiniPushT
from tiny_vla import (
    TinyVLA, ScriptedExpert, collect_demos, PushTDataset, train, evaluate
)
import torch
import matplotlib.pyplot as plt
print('Imports OK')

## 1. Explore the Environment

In [ ]:
env = MiniPushT()
obs, info = env.reset(seed=42)
print('Observation shape:', obs.shape)
print('Info keys:', list(info.keys()))
plt.imshow(obs)
plt.title('MiniPushT observation')
plt.axis('off')
plt.show()

## 2. Collect Expert Demonstrations

In [ ]:
expert = ScriptedExpert()
print('Collecting 1000 demos...')
demos = collect_demos(env, expert, n_episodes=1000)
print(f'Collected {len(demos)} transitions')
print('Sample:', {k: v.shape if hasattr(v, "shape") else v for k, v in demos[0].items()})

## 3. Train the Model

In [ ]:
dataset = PushTDataset(demos)
model = TinyVLA()
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')
losses = train(model, dataset, epochs=20, batch_size=64)
plt.plot(losses)
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training loss')
plt.show()

## 4. Evaluate

In [ ]:
rate = evaluate(model, env, n_episodes=50)
print(f'Success rate: {rate * 100:.1f}%')
print('Expected: ~70-80%')

## Appendix: Noisy Expert

What happens when training data comes from a noisy expert?

In [ ]:
from tiny_vla import NoisyExpert
noisy = NoisyExpert(expert, epsilon=0.1)
noisy_demos = collect_demos(env, noisy, n_episodes=1000)
noisy_model = TinyVLA()
train(noisy_model, PushTDataset(noisy_demos), epochs=20, batch_size=64)
noisy_rate = evaluate(noisy_model, env, n_episodes=50)
print(f'Noisy expert success rate: {noisy_rate * 100:.1f}%')
print(f'Clean expert success rate: {rate * 100:.1f}%')